In [ ]:
import os
import pandas as pd
import yfinance as yf

os.chdir(os.path.dirname(os.getcwd()))


from src.constants.config import path_portfolio
from src.constants.tickers import isin_to_ticker

In [ ]:
# load portfolio
df = pd.read_csv(path_portfolio)

df['Ticker'] = df['ISIN'].replace(isin_to_ticker)
df['SnapshotDate'] = pd.to_datetime(df['SnapshotDate']).dt.date

df_grouped = df.groupby(['Product', 'ISIN', 'Ticker']).agg({'SnapshotDate': ['min', 'max']})
df_grouped.columns = ['StartDate', 'EndDate']
df_grouped = df_grouped.reset_index()

In [ ]:
# Collect stockrates
all_data = []
empty_data = []

for index, row in df_grouped.iterrows():
    start_date = row['StartDate']
    end_date = row['EndDate']
    symbol = row['Ticker']
    product = row['Product']
    isincode = row['ISIN']
    

    print(f"Fetching data for: {symbol} ({product})")

    try:
        ticker = yf.Ticker(symbol)
        currency = ticker.info['currency']
        history_df = (
            ticker.history(start=start_date, end=end_date)
            .reset_index()[['Date', 'Close']]
            .assign(Product=product, ISIN=isincode, Ticker=symbol, currency=currency)
        )

        if history_df.shape[0] == 0:
            print(f"No data found for {symbol}")
            empty_data.append({
                "Ticker": symbol,
                "Product": product,
                "ISIN": isincode,
                "StartDate": start_date,
                "EndDate": end_date
            })
        else:
            print(f"Retrieved {history_df.shape[0]} rows for {symbol}")
            all_data.append(history_df)

    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        empty_data.append({
            "Ticker": symbol,
            "Product": product,
            "ISIN": isincode,
            "StartDate": start_date,
            "EndDate": end_date,
            "Error": str(e)
        })

# Combine successful results
combined_df = pd.concat(all_data, ignore_index=True)



In [ ]:
# Collect currencies
currencies = ['EURUSD=X', 'GBPEUR=X']
currency_data = []

for c in currencies:
    print(f"Fetching currency data for: {c}")
    
    # Detect currency type
    if 'GBP' in c:
        currency = 'GBP'
    elif 'USD' in c:
        currency = 'USD'
    else:
        currency = 'Unknown'

    try:
        ticker = yf.Ticker(c)
        cur_df = (
            ticker.history(period='max')
            .reset_index()[['Date', 'Close']]
            .assign(Product=c, Currency=currency)
            .rename({'Close':'Fx_rate'},axis=1)
        )

        print(f"Retrieved {cur_df.shape[0]} rows for {c}")
        currency_data.append(cur_df)

    except Exception as e:
        print(f"Error fetching data for {c}: {e}")

combined_currency_df = pd.concat(currency_data, ignore_index=True)
combined_currency_df['Date'] = pd.to_datetime(combined_currency_df['Date']).dt.date